# SafeSync: Drug Information Retrieval and Prediction Logic

## Project Goal
This notebook implements the logic for the Medical Professional's Rapid Reference (SafeSync) tool. The goal is to input a **Drug Name (API)** and **Dosage Form** and retrieve a comprehensive summary.

## Strategy: Retrieval-First, ML-Fallback
We will implement a robust two-stage pipeline:
1.  **Stage A (Deterministic Retrieval):** If the exact (API, Dosage Form) pair exists in our cleaned dataset, we will retrieve its information directly.
2.  **Stage B (ML Fallback):** If the pair is not found, we will use Random Forest models to predict the likely formulation and storage conditions.

In [12]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 200)

In [13]:
# --- Load Datasets ---
try:
    df_drugs = pd.read_csv('datasets/drug/merged.csv')
    df_interactions = pd.read_csv('datasets/drug/Interaction.csv')
    print("Datasets loaded successfully.")
except FileNotFoundError:
    print("ERROR: Make sure 'merged.csv' and 'Interaction.csv' are in the 'datasets/drug/' folder.")
    exit()

# --- 1. Clean Column Names and Text Fields ---
df_drugs.columns = df_drugs.columns.str.strip()
df_interactions.columns = df_interactions.columns.str.strip()
df_interactions.rename(columns={'Mechansim': 'Mechanism'}, inplace=True)

# Unify case to Title Case for consistency
for col in df_drugs.select_dtypes(['object']).columns:
    df_drugs[col] = df_drugs[col].str.strip().str.title()
for col in df_interactions.select_dtypes(['object']).columns:
    df_interactions[col] = df_interactions[col].str.strip().str.title()

# --- 2. Tokenize Formulation ---
df_drugs.dropna(subset=['Excipient_List', 'Excipient_Roles'], inplace=True)

def process_formulation(row):
    excipients = [e.strip() for e in row['Excipient_List'].split(';') if e.strip().upper() != 'API']
    roles = [r.strip() for r in row['Excipient_Roles'].split(';') if r.strip().upper() != 'ACTIVE PHARMACEUTICAL INGREDIENT']
    min_len = min(len(excipients), len(roles))
    return excipients[:min_len], roles[:min_len]

formulations = df_drugs.apply(process_formulation, axis=1)
df_drugs['Excipients_Clean'] = [f[0] for f in formulations]
df_drugs['Roles_Clean'] = [f[1] for f in formulations]

# --- 3. Deduplicate by taking the most frequent value (mode) ---
grouped = df_drugs.groupby(['Active pharmaceutical Ingredient (API)', 'Dosage_Form'])
mode_agg = lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
df_clean = grouped.agg({
    'Therapeutic_Class': mode_agg,
    'Excipients_Clean': 'first',
    'Roles_Clean': 'first',
    'Recommended_Storage': mode_agg,
    'Regulatory_Notes': mode_agg,
    'ReferenceSource': mode_agg
}).reset_index()

print("--- Data Cleaning Complete ---")
print(f"Total unique drug formulations: {len(df_clean)}")
display(df_clean.head())

Datasets loaded successfully.
--- Data Cleaning Complete ---
Total unique drug formulations: 6170


,Active pharmaceutical Ingredient (API),Dosage_Form,Therapeutic_Class,Excipients_Clean,Roles_Clean,Recommended_Storage,Regulatory_Notes,ReferenceSource
0,Alendronate,Ear Drops,General Medicine,"[Purified Water Or Oil Base, Preservative, Buffer, Viscosity Agent]","[Exipient (Various), Preservative, Buffering Agent, Exipient (Various)]",Store At 2-8Â°C (Refrigerate),NaN,Rowe (Handbook Of Pharmaceutical Excipients); Lachman (Theory & Practice); Ich Q1A; Fda Iid
1,Alendronate,Eye Drops,General Medicine,"[Purified Water, Buffer (Citrate/Phosphate), Tonicity Agent (Sodium Chloride), Preservative (Benzalkonium), Viscosity Agent (Cmc)]","[Exipient (Various), Exipient (Various), Exipient (Various), Exipient (Various), Exipient (Various)]","Store Below 25Â°C, Protect From Light",NaN,Rowe (Handbook Of Pharmaceutical Excipients); Lachman (Theory & Practice); Ich Q1A; Fda Iid
2,Alendronate,Inhalation Mdi,General Medicine,"[Propellant (Hfa), Co-Solvent (Ethanol), Surfactant (Oleic Acid), Valve Components]","[Propellant, Exipient (Various), Exipient (Various), Exipient (Various)]",Store At 2-8Â°C (Refrigerate),Use Antioxidant If Peroxidation Risk,Rowe (Handbook Of Pharmaceutical Excipients); Lachman (Theory & Practice); Ich Q1A; Fda Iid
3,Alendronate,Oral Solution,General Medicine,"[Water For Injection / Purified Water, Propylene Glycol, Glycerin, Sorbitol, Preservative (Sodium Benzoate), Flavor, Buffer]","[Vehicle, Solvent / Humectant, Humectant / Plasticizer, Exipient (Various), Exipient (Various), Flavoring Agent, Buffering Agent]","Room Temperature, Protect From Moisture",Avoid Strong Oxidizers,Rowe (Handbook Of Pharmaceutical Excipients); Lachman (Theory & Practice); Ich Q1A; Fda Iid
4,Alendronate,Sachet Powder,General Medicine,"[Mannitol / Dextrose, Flavor, Anti-Caking Agent (Silicon Dioxide), Citric Acid, Preservative (If Needed)]","[Diluent / Sweetener, Flavoring Agent, Exipient (Various), Ph Adjuster, Exipient (Various)]","Store Below 25Â°C, Protect From Light",NaN,Rowe (Handbook Of Pharmaceutical Excipients); Lachman (Theory & Practice); Ich Q1A; Fda Iid


This function is the core of our system. It performs a direct lookup for any drug combination that exists in our clean dataset, providing the most accurate information.

In [14]:
def get_drug_info_retrieval(api, dosage_form, df_data, df_interact):
    api_norm = api.strip().title()
    dosage_form_norm = dosage_form.strip().title()
    
    record = df_data[(df_data['Active pharmaceutical Ingredient (API)'] == api_norm) &
                     (df_data['Dosage_Form'] == dosage_form_norm)]

    if record.empty:
        return None

    record = record.iloc[0]
    result = {
        "source": "Exact Match from Dataset",
        "formulation": pd.DataFrame({'Excipient': record['Excipients_Clean'], 'Role': record['Roles_Clean']}),
        "storage": record['Recommended_Storage'],
        "compatibility_notes": record['Regulatory_Notes'],
        "references": record['ReferenceSource'],
        "interactions": df_interact[df_interact['Drug A'] == api_norm]
    }
    return result

This stage is for new or unseen drug combinations. We train two simple Random Forest models: one to predict storage conditions and another to predict the likely excipients in the formulation.

In [15]:
print("--- Training ML Fallback Models ---")

# --- Model 1: Storage Prediction ---
df_storage = df_clean.dropna(subset=['Recommended_Storage', 'Therapeutic_Class'])
X_s = df_storage[['Active pharmaceutical Ingredient (API)', 'Dosage_Form', 'Therapeutic_Class']]
y_s = df_storage['Recommended_Storage']

# Create and fit encoders
api_encoder_s = LabelEncoder().fit(X_s['Active pharmaceutical Ingredient (API)'])
dosage_encoder_s = LabelEncoder().fit(X_s['Dosage_Form'])
class_encoder_s = LabelEncoder().fit(X_s['Therapeutic_Class'])
storage_encoder = LabelEncoder().fit(y_s)

# Transform data
X_s_encoded = pd.DataFrame({
    'API': api_encoder_s.transform(X_s['Active pharmaceutical Ingredient (API)']),
    'Dosage': dosage_encoder_s.transform(X_s['Dosage_Form']),
    'Class': class_encoder_s.transform(X_s['Therapeutic_Class'])
})
y_s_encoded = storage_encoder.transform(y_s)

storage_model = RandomForestClassifier(n_estimators=100, random_state=42)
storage_model.fit(X_s_encoded, y_s_encoded)
print("Storage prediction model trained successfully.")

# --- Model 2: Formulation (Excipient) Prediction ---
df_formulation = df_clean[df_clean['Excipients_Clean'].apply(len) > 0]
X_f = df_formulation[['Active pharmaceutical Ingredient (API)', 'Dosage_Form', 'Therapeutic_Class']]
y_f = df_formulation['Excipients_Clean']

# Create and fit encoders
api_encoder_f = LabelEncoder().fit(X_f['Active pharmaceutical Ingredient (API)'])
dosage_encoder_f = LabelEncoder().fit(X_f['Dosage_Form'])
class_encoder_f = LabelEncoder().fit(X_f['Therapeutic_Class'])
mlb = MultiLabelBinarizer().fit(y_f)

# Transform data
X_f_encoded = pd.DataFrame({
    'API': api_encoder_f.transform(X_f['Active pharmaceutical Ingredient (API)']),
    'Dosage': dosage_encoder_f.transform(X_f['Dosage_Form']),
    'Class': class_encoder_f.transform(X_f['Therapeutic_Class'])
})
y_f_encoded = mlb.transform(y_f)

formulation_model = MultiOutputClassifier(RandomForestClassifier(n_estimators=50, random_state=42))
formulation_model.fit(X_f_encoded, y_f_encoded)
print("Formulation prediction model trained successfully.")

--- Training ML Fallback Models ---
Storage prediction model trained successfully.
Formulation prediction model trained successfully.


In [16]:
def get_drug_info_full_pipeline(api, dosage_form, df_data, df_interact):
    retrieval_result = get_drug_info_retrieval(api, dosage_form, df_data, df_interact)
    if retrieval_result is not None:
        return retrieval_result
    
    print(f"\nINFO: No exact match for '{api} {dosage_form}'. Using ML Fallback to predict.")
    
    api_norm = api.strip().title()
    therapeutic_class_series = df_data[df_data['Active pharmaceutical Ingredient (API)'] == api_norm]['Therapeutic_Class']
    
    if therapeutic_class_series.empty:
        return {"source": "Prediction Failed", "reason": f"Cannot predict, as the API '{api}' is completely unknown."}
    
    therapeutic_class = therapeutic_class_series.iloc[0]

    # --- Predict Storage ---
    storage_input = pd.DataFrame({
        'API': [api_encoder_s.transform([api_norm])[0]],
        'Dosage': [dosage_encoder_s.transform([dosage_form.title()])[0]],
        'Class': [class_encoder_s.transform([therapeutic_class])[0]]
    })
    predicted_storage_encoded = storage_model.predict(storage_input)
    predicted_storage = storage_encoder.inverse_transform(predicted_storage_encoded)[0]

    # --- Predict Formulation ---
    formulation_input = pd.DataFrame({
        'API': [api_encoder_f.transform([api_norm])[0]],
        'Dosage': [dosage_encoder_f.transform([dosage_form.title()])[0]],
        'Class': [class_encoder_f.transform([therapeutic_class])[0]]
    })
    predicted_excipients_encoded = formulation_model.predict(formulation_input)
    predicted_excipients = mlb.inverse_transform(predicted_excipients_encoded)[0]
    
    result = {
        "source": "ML Prediction (Fallback)",
        "formulation": pd.DataFrame({'Excipient': predicted_excipients, 'Role': ["(Predicted)" for _ in predicted_excipients]}),
        "storage": predicted_storage,
        "compatibility_notes": "Use standard compatibility guidelines; data is predicted.",
        "references": "General pharmaceutical handbooks (e.g., Rowe, Lachman).",
        "interactions": df_interact[df_interact['Drug A'] == api_norm]
    }
    return result

This final function combines both stages into a single, easy-to-use pipeline. It tries to retrieve data first and, if it fails, uses the trained ML models to generate a prediction.

In [18]:
# ===============================================================
#          INTERACTIVE TEST: ENTER YOUR VALUES HERE
# ===============================================================

# --- Test Case 1: A drug that exists in the dataset ---
test_api_1 = "Warfarin"
test_dosage_1 = "Sr Tablet"

# --- Test Case 2: A new combination to trigger the ML model ---
test_api_2 = "Aspirin"
test_dosage_2 = "Inhalation MDI"


# ===============================================================
#                 RUN THIS CELL TO SEE THE RESULTS
# ===============================================================

print("--- RUNNING TEST CASE 1 (Exact Match) ---")
final_result_1 = get_drug_info_full_pipeline(test_api_1, test_dosage_1, df_clean, df_interactions)
if final_result_1:
    print(f"Source: {final_result_1['source']}")
    print(f"Storage: {final_result_1['storage']}")
    print("Formulation:")
    display(final_result_1['formulation'])
    print("Interactions:")
    display(final_result_1['interactions'][['Drug A', 'Drug B', 'Severity', 'Mechanism']])

print("\n" + "="*50 + "\n")

print("--- RUNNING TEST CASE 2 (ML Fallback) ---")
final_result_2 = get_drug_info_full_pipeline(test_api_2, test_dosage_2, df_clean, df_interactions)
if final_result_2:
    print(f"Source: {final_result_2['source']}")
    if final_result_2['source'] == "Prediction Failed":
        print(f"Reason: {final_result_2['reason']}")
    else:
        print(f"Predicted Storage: {final_result_2['storage']}")
        print("Predicted Formulation:")
        display(final_result_2['formulation'])
        print("Interactions (Lookup):")
        display(final_result_2['interactions'][['Drug A', 'Drug B', 'Severity', 'Mechanism']])

--- RUNNING TEST CASE 1 (Exact Match) ---
Source: Exact Match from Dataset
Storage: Store Below 30Â°C, Avoid Humidity
Formulation:


,Excipient,Role
0,Hpmc,Binder / Controlled Release Polymer / Film Former
1,Ethylcellulose,Coating Agent / Controlled Release Polymer
2,Microcrystalline Cellulose (Mcc),Diluent / Binder
3,Magnesium Stearate,Lubricant
4,Peg 6000,Plasticizer / Binder
5,Talc,Anti-Adherent / Glidant
6,Coating Polymer,Coating / Polymer


Interactions:


,Drug A,Drug B,Severity,Mechanism
980,Warfarin,Aspirin,Major,Increased Bleeding Risk - Additive Anticoagulant/Antiplatelet Effect
981,Warfarin,Amoxicillin,Major,Potential Increase In Inr - Antibiotic Effect On Gut Flora
982,Warfarin,Metronidazole,Major,Increased Bleeding Via Cyp Inhibition/Altered Gut Flora
983,Warfarin,Fluconazole,Major,Cyp Inhibition Leading To Increased Warfarin Levels
984,Warfarin,Amiodarone,Major,Pharmacokinetic Interaction Increasing Warfarin Levels
985,Warfarin,Salmeterol,Major,High-Risk Pharmacokinetic Or Pharmacodynamic Interaction (See References)
986,Warfarin,Beclometasone,Major,High-Risk Pharmacokinetic Or Pharmacodynamic Interaction (See References)
987,Warfarin,Antacid (Omeprazole),Major,High-Risk Pharmacokinetic Or Pharmacodynamic Interaction (See References)
988,Warfarin,Saxagliptin,Major,High-Risk Pharmacokinetic Or Pharmacodynamic Interaction (See References)
989,Warfarin,Glimepiride,Major,High-Risk Pharmacokinetic Or Pharmacodynamic Interaction (See References)




--- RUNNING TEST CASE 2 (ML Fallback) ---
Source: Exact Match from Dataset
Predicted Storage: Store At 2-8Â°C (Refrigerate)
Predicted Formulation:


,Excipient,Role
0,Propellant (Hfa),Propellant
1,Co-Solvent (Ethanol),Exipient (Various)
2,Surfactant (Oleic Acid),Exipient (Various)
3,Valve Components,Exipient (Various)


Interactions (Lookup):


,Drug A,Drug B,Severity,Mechanism
66,Aspirin,Citalopram,NaN,No Significant Interaction Identified In Checked List
67,Aspirin,Ampicillin,Minor,Minor Interaction — Usually Clinically Manageable
68,Aspirin,Iron (Ferrous Sulfate),Minor,Minor Interaction — Usually Clinically Manageable
69,Aspirin,Captopril,Moderate,Moderate Interaction — Monitor Therapy
70,Aspirin,Turmeric (Curcumin),Moderate,Moderate Interaction — Monitor Therapy
71,Aspirin,Canagliflozin,Moderate,Moderate Interaction — Monitor Therapy
72,Aspirin,Risperidone,Minor,Minor Interaction — Usually Clinically Manageable
73,Aspirin,Sertraline,Minor,Minor Interaction — Usually Clinically Manageable
